## Imports

In [1]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px

from sklearn.decomposition import PCA

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

import PyWGCNA

## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
DESEQ_DIR = Path("../../data/interim/deseq2")

LOGCPM_PATH = PROCESSED_DIR / "microplastic_logcpm_filtered.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"
# Input: log2(normed_counts + 1) gerado pelo notebook 002
WGCNA_INPUT_PATH = PROCESSED_DIR / "microplastic_log2norm_wgcna_input.csv"

# Saídas
WGCNA_DIR = Path("../../data/interim/wgcna")
WGCNA_DIR.mkdir(parents=True, exist_ok=True)

GENE_MODULES_PATH              = WGCNA_DIR / "wgcna_gene_modules.csv"
MODULE_EIGENGENES_PATH         = WGCNA_DIR / "wgcna_module_eigengenes.csv"
MODULE_EIGENGENES_TREATED_PATH = WGCNA_DIR / "wgcna_module_eigengenes_treated.csv"
MODULE_TRAIT_CORR_PATH         = WGCNA_DIR / "wgcna_module_trait_correlations_treated.csv"
MODULE_TRAIT_PVAL_PATH         = WGCNA_DIR / "wgcna_module_trait_pvalues_treated.csv"
MODULE_TRAIT_PADJ_PATH         = WGCNA_DIR / "wgcna_module_trait_padj_treated.csv"
MODULE_SUMMARY_PATH            = WGCNA_DIR / "wgcna_module_summary.csv"
HEATMAP_PATH                   = WGCNA_DIR / "wgcna_module_trait_heatmap_treated.png"

# Parâmetros-chave do WGCNA — altere aqui para reexecutar com diferentes valores.
# O nome do modelo e o path do cache são derivados automaticamente dos parâmetros,
# evitando sobrescrever resultados anteriores.
MIN_MODULE_SIZE = 20
ME_DISS_THRES   = 0.25

WGCNA_NAME       = f"microplastic_wgcna_minmod{MIN_MODULE_SIZE}_meds{int(ME_DISS_THRES * 100)}"
WGCNA_CACHE_PATH = WGCNA_DIR / f"{WGCNA_NAME}.p"

## Carregamento dos Dados de DEG

In [3]:
log2norm_df = pd.read_csv(WGCNA_INPUT_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("log2norm:", log2norm_df.shape)
print("metadata:", metadata_df.shape)

display(log2norm_df.head())
display(metadata_df.head())

log2norm: (12176, 25)
metadata: (24, 9)


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MB1_1,MB1_2,MB1_3,...,MD1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,8.495462,8.450787,8.522997,8.361727,7.986636,8.218316,8.495056,8.136695,8.608448,...,8.256791,8.322836,8.369257,8.097211,8.272110,8.003671,8.616512,8.234527,8.321817,8.461644
1,ENSG00000000419,9.602250,8.919860,9.676560,9.491607,9.459724,9.228012,9.289487,9.577056,9.501974,...,9.781742,9.421966,9.663061,9.276540,9.361926,9.593103,9.537132,9.461464,9.762925,9.298947
2,ENSG00000000457,4.818618,6.950711,5.152466,4.995209,5.243740,4.919546,5.047678,5.071817,5.001960,...,5.447777,4.882649,5.203948,2.789318,4.416671,5.488284,5.716265,5.347044,5.944565,5.911186
3,ENSG00000000460,5.259300,4.897628,3.138396,2.524291,2.720141,4.958863,4.359242,5.551225,5.479900,...,0.000000,4.835425,2.160350,0.000000,4.064434,3.442941,4.540423,3.657667,4.300918,4.913981
4,ENSG00000001036,9.949776,9.976058,10.080300,9.906739,9.953377,10.599921,10.073621,10.080813,9.783203,...,10.036049,10.022790,9.818998,9.955213,9.849623,9.953890,10.260971,10.283438,10.236628,10.253601


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.0,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.0,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.0,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.1,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.1,False,treated,MA1_2,MA1,2


## Construção do WGCNA

In [4]:
# Preparação da matriz de expressão para PyWGCNA.
# A matriz em arquivo está em genes x amostras.
# Para o construtor WGCNA do PyWGCNA, deve-se usar amostras x genes.

sample_ids = metadata_df["sample_id"].tolist()
expr_sample_cols = [c for c in log2norm_df.columns if c != "gene_id"]

missing_in_expr = sorted(set(sample_ids) - set(expr_sample_cols))
missing_in_meta = sorted(set(expr_sample_cols) - set(sample_ids))

if missing_in_expr or missing_in_meta:
    raise ValueError(
        f"Inconsistência entre expressão e metadata.\n"
        f"Ausentes na expressão: {missing_in_expr}\n"
        f"Ausentes no metadata: {missing_in_meta}"
    )

# Reordena as colunas conforme a ordem do metadata
log2norm_df = log2norm_df[["gene_id"] + sample_ids]

# Formato esperado para WGCNA: amostras x genes
expr_wgcna = log2norm_df.set_index("gene_id").T
expr_wgcna.index.name = "sample_id"

# Metadata alinhado
metadata = metadata_df.set_index("sample_id").loc[expr_wgcna.index].copy()

print("Matriz para WGCNA:", expr_wgcna.shape)
print("Metadata alinhado:", metadata.shape)

display(expr_wgcna.iloc[:5, :5])
display(metadata.head())

Matriz para WGCNA: (24, 12176)
Metadata alinhado: (24, 8)


gene_id,ENSG00000000003,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000001036
sample_id,,,,,
CTR_1,8.495462,9.602250,4.818618,5.259300,9.949776
CTR_2,8.450787,8.919860,6.950711,4.897628,9.976058
CTR_3,8.522997,9.676560,5.152466,3.138396,10.080300
MA1_1,8.361727,9.491607,4.995209,2.524291,9.906739
MA1_2,7.986636,9.459724,5.243740,2.720141,9.953377


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2


In [5]:
# Preparação do metadata para PyWGCNA.

# Corrige possíveis problemas de tipo ao ler CSV
if metadata["is_control"].dtype == object:
    metadata["is_control"] = metadata["is_control"].astype(str).str.lower().map({
        "true": True,
        "false": False
    })

metadata["particle_size_um"] = pd.to_numeric(metadata["particle_size_um"], errors="coerce")
metadata["particle_size_nm"] = pd.to_numeric(metadata["particle_size_nm"], errors="coerce")
metadata["concentration_gL"] = pd.to_numeric(metadata["concentration_gL"], errors="coerce")

display(metadata.dtypes)
display(metadata.head())

particle_type        object
particle_size_um    float64
particle_size_nm    float64
concentration_gL    float64
is_control             bool
treatment_status     object
group                object
replicate             int64
dtype: object

,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2


In [6]:
# Construção do objeto WGCNA

pyw = PyWGCNA.WGCNA(
    name=WGCNA_NAME,                 # Derivado dos parâmetros — veja Cell 3
    species="human",                 # Usado para anotação funcional
    geneExp=expr_wgcna,              # Matriz de expressão: amostras × genes (log2norm)
    sampleInfo=metadata,             # Metadata indexado por sample_id
    outputPath=str(WGCNA_DIR) + "/", # Trailing slash obrigatório — PyWGCNA concatena strings
    save=True,                       # Salva resultados intermediários automaticamente

    # Filtro de expressão mínima — já filtramos no notebook 001, então desativado aqui
    TPMcutoff=0,

    # Expoentes a testar para escolha do soft-threshold β (escala 1–10 contínuo, 12–20 de 2 em 2)
    # O WGCNA seleciona o menor expoente que atinge RsquaredCut na topologia livre de escala
    powers=list(range(1, 11)) + list(range(12, 22, 2)),
    RsquaredCut=0.8,                     # R² mínimo do ajuste de topologia livre de escala
    MeanCut=100,                         # conectividade média máxima (evita redes super-conectadas)

    # "signed hybrid": preserva o sinal da correlação (positivo ≠ negativo), mas trata
    # correlações negativas de forma mais suave que "signed" puro — padrão recomendado
    # para dados de RNA-seq onde co-regulação negativa é biologicamente relevante
    networkType="signed hybrid",
    TOMType="signed",                    # TOM com sinal: considera direção da correlação no overlap

    # Reduzir esse valor permite que clusters menores formem módulos próprios.
    minModuleSize=MIN_MODULE_SIZE,

    # Módulos com correlação entre eigengenes > 1 - ME_DISS_THRES são fundidos.
    # Valor ligeiramente maior (0.25) = fusão menos agressiva → mais módulos distintos preservados.
    MEDissThres=ME_DISS_THRES,
)

Saving data to be True, checking requirements ...


In [7]:
if WGCNA_CACHE_PATH.exists():
    print(f"Carregando WGCNA do cache: {WGCNA_CACHE_PATH}")
    with open(WGCNA_CACHE_PATH, "rb") as f:
        pyw = pickle.load(f)
    print("WGCNA carregado com sucesso.")
else:
    print("Cache não encontrado — executando WGCNA (pode levar vários minutos)...")
    pyw.runWGCNA()
    if hasattr(pyw, "saveWGCNA"):
        pyw.saveWGCNA()
    print(f"WGCNA executada e cache salvo em: {WGCNA_CACHE_PATH}")

Carregando WGCNA do cache: ../../data/interim/wgcna/microplastic_wgcna_minmod20_meds25.p
WGCNA carregado com sucesso.


In [8]:
# Diagnóstico do soft-thresholding power selecionado pelo PyWGCNA
print(f"Soft-thresholding power selecionado: {pyw.power}")
print(f"(R² target: {pyw.RsquaredCut})")

Soft-thresholding power selecionado: 4
(R² target: 0.8)


## Inspeção dos Resultados

In [9]:
# Verificar os principais atributos do objeto PyWGCNA após a execução

print("type(pyw.datExpr):", type(getattr(pyw, "datExpr", None)))
print("shape datExpr:", getattr(getattr(pyw, "datExpr", None), "shape", None))

if hasattr(pyw, "datExpr") and pyw.datExpr is not None:
    print("var columns:", pyw.datExpr.var.columns)
    display(pyw.datExpr.var.head())

type(pyw.datExpr): <class 'anndata._core.anndata.AnnData'>
shape datExpr: (24, 12176)
var columns: Index(['dynamicColors', 'moduleColors', 'moduleLabels'], dtype='object')


,dynamicColors,moduleColors,moduleLabels
gene_id,,,
ENSG00000000003,dimgrey,dimgrey,10
ENSG00000000419,mistyrose,mistyrose,17
ENSG00000000457,dimgrey,dimgrey,10
ENSG00000000460,dimgrey,dimgrey,10
ENSG00000001036,indianred,indianred,13


In [10]:
# Extração de genes e módulos finais obtidos com WGCNA

if not hasattr(pyw, "datExpr") or pyw.datExpr is None:
    raise ValueError("pyw.datExpr não está disponível.")

gene_var = pyw.datExpr.var.copy()

expected_cols = {"dynamicColors", "moduleColors", "moduleLabels"}
missing_cols = expected_cols - set(gene_var.columns)

if missing_cols:
    raise ValueError(f"Colunas esperadas ausentes em pyw.datExpr.var: {missing_cols}")

gene_modules_df = (
    gene_var
    .reset_index()
    .rename(columns={"index": "gene_id"})
    [["gene_id", "dynamicColors", "moduleColors", "moduleLabels"]]
    .copy()
)

gene_modules_df = gene_modules_df.rename(columns={
    "dynamicColors": "dynamic_module",
    "moduleColors": "module",
    "moduleLabels": "module_label"
})

gene_modules_df.to_csv(GENE_MODULES_PATH, index=False)

print("Tabela gene -> módulo salva em:", GENE_MODULES_PATH)
print("Dimensões:", gene_modules_df.shape)

display(gene_modules_df.head())
display(
    gene_modules_df["module"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "module", "module": "n_genes"})
)

Tabela gene -> módulo salva em: ../../data/interim/wgcna/wgcna_gene_modules.csv
Dimensões: (12176, 4)


,gene_id,dynamic_module,module,module_label
0,ENSG00000000003,dimgrey,dimgrey,10
1,ENSG00000000419,mistyrose,mistyrose,17
2,ENSG00000000457,dimgrey,dimgrey,10
3,ENSG00000000460,dimgrey,dimgrey,10
4,ENSG00000001036,indianred,indianred,13


,n_genes,count
0,dimgrey,3883
1,darkgrey,3266
2,mistyrose,2130
3,gainsboro,1124
4,maroon,347
5,white,317
6,indianred,216
7,brown,158
8,firebrick,139
9,red,125


In [11]:
# Cálculo dos module eigengenes a partir dos módulos finais.
#
# NOTA: Calculamos eigengenes manualmente via PCA ao invés de usar pyw.getEigengenes()
# porque a API do PyWGCNA retorna eigengenes na orientação interna do objeto AnnData,
# dificultando o alinhamento com nosso metadata indexado por sample_id.
# O resultado é matematicamente equivalente: o 1º componente principal do bloco de
# expressão de cada módulo, com sinal corrigido para correlacionar positivamente com
# o perfil médio do módulo.

module_eigengenes = {}

for module_name, module_df in gene_modules_df.groupby("module"):
    genes = [g for g in module_df["gene_id"].tolist() if g in expr_wgcna.columns]

    if len(genes) == 0:
        continue

    X = expr_wgcna[genes].copy()

    # Se o módulo tiver apenas 1 gene, usa o próprio perfil
    if X.shape[1] == 1:
        eigengene = X.iloc[:, 0].values.astype(float)
    else:
        pca = PCA(n_components=1, random_state=42)
        eigengene = pca.fit_transform(X.values).ravel()

        # Ajusta o sinal para ficar coerente com o perfil médio do módulo
        mean_profile = X.mean(axis=1).values
        corr_sign = np.corrcoef(eigengene, mean_profile)[0, 1]
        if pd.notna(corr_sign) and corr_sign < 0:
            eigengene = -eigengene

    module_eigengenes[f"ME_{module_name}"] = eigengene

module_eigengenes_df = pd.DataFrame(
    module_eigengenes,
    index=expr_wgcna.index
)

module_eigengenes_df.index.name = "sample_id"
module_eigengenes_df.to_csv(MODULE_EIGENGENES_PATH)

print("Module eigengenes salvos em:", MODULE_EIGENGENES_PATH)
print("Dimensões:", module_eigengenes_df.shape)

display(module_eigengenes_df.head())

Module eigengenes salvos em: ../../data/interim/wgcna/wgcna_module_eigengenes.csv
Dimensões: (24, 29)


,ME_antiquewhite,ME_bisque,ME_blanchedalmond,ME_brown,ME_burlywood,ME_chocolate,ME_coral,ME_darkgrey,ME_darkorange,ME_darksalmon,...,ME_orangered,ME_peachpuff,ME_peru,ME_red,ME_saddlebrown,ME_sandybrown,ME_seashell,ME_sienna,ME_tan,ME_white
sample_id,,,,,,,,,,,,,,,,,,,,,
CTR_1,-0.375386,-1.637375,1.021909,0.387471,-1.306193,1.434195,0.567610,-10.196532,1.226707,-0.097652,...,0.336386,3.127817,-0.073363,-0.307537,0.037556,1.694046,-1.116049,0.646524,0.787013,-0.549383
CTR_2,-3.516172,-0.404852,-0.620645,-1.454856,-3.130498,3.304383,1.077420,-16.035924,0.788736,-1.496775,...,-1.581673,-0.703774,4.412278,2.559447,2.861798,-1.660803,1.934903,-5.278875,3.716757,-11.175795
CTR_3,-0.236977,3.993195,-3.561609,0.645962,-3.260311,-8.150420,-2.025751,-17.803953,0.206924,-2.059646,...,-0.968266,-0.601913,-1.072723,-1.093270,1.053037,0.034980,1.646322,-4.148310,-0.324214,-7.387820
MA1_1,1.869365,-0.892133,2.513553,0.350257,0.453260,-3.256164,1.170604,0.139914,1.520527,0.878495,...,-5.998327,3.351834,3.029996,1.322148,0.887940,-2.262799,1.650379,2.249196,1.595445,-1.689926
MA1_2,1.198301,-1.295283,2.005308,2.453625,2.719077,2.005403,-2.571877,1.976873,-2.085860,-2.188630,...,-0.654931,-2.649666,-6.092502,2.346185,0.844902,-2.001848,1.884629,2.057451,-3.496913,-1.187131


In [12]:
# Filtrar apenas as amostras tratadas para correlação módulo-traço

metadata_treated = metadata[metadata["is_control"] == False].copy()
module_eigengenes_treated = module_eigengenes_df.loc[metadata_treated.index].copy()

print("Amostras totais:", metadata.shape[0])
print("Amostras tratadas:", metadata_treated.shape[0])
print("Module eigengenes (tratadas):", module_eigengenes_treated.shape)

display(metadata_treated.head())
display(module_eigengenes_treated.head())

Amostras totais: 24
Amostras tratadas: 21
Module eigengenes (tratadas): (21, 29)


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
MA1_1,polystyrene,1.0,1000.0,0.10,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.10,False,treated,MA1,2
MA1_3,polystyrene,1.0,1000.0,0.10,False,treated,MA1,3
MB1_1,polystyrene,1.0,1000.0,0.01,False,treated,MB1,1
MB1_2,polystyrene,1.0,1000.0,0.01,False,treated,MB1,2


,ME_antiquewhite,ME_bisque,ME_blanchedalmond,ME_brown,ME_burlywood,ME_chocolate,ME_coral,ME_darkgrey,ME_darkorange,ME_darksalmon,...,ME_orangered,ME_peachpuff,ME_peru,ME_red,ME_saddlebrown,ME_sandybrown,ME_seashell,ME_sienna,ME_tan,ME_white
sample_id,,,,,,,,,,,,,,,,,,,,,
MA1_1,1.869365,-0.892133,2.513553,0.350257,0.453260,-3.256164,1.170604,0.139914,1.520527,0.878495,...,-5.998327,3.351834,3.029996,1.322148,0.887940,-2.262799,1.650379,2.249196,1.595445,-1.689926
MA1_2,1.198301,-1.295283,2.005308,2.453625,2.719077,2.005403,-2.571877,1.976873,-2.085860,-2.188630,...,-0.654931,-2.649666,-6.092502,2.346185,0.844902,-2.001848,1.884629,2.057451,-3.496913,-1.187131
MA1_3,0.926038,1.330926,-1.121827,3.073040,1.582117,-0.614648,2.642839,-0.900120,-0.248484,1.087712,...,-1.248563,3.562182,0.961915,-0.398990,0.839033,-0.909535,0.576642,1.653385,1.581869,7.804055
MB1_1,2.685035,-3.073713,2.233145,0.953819,1.935786,0.367047,-1.417240,15.465638,-2.447460,1.766154,...,-2.904673,1.827023,2.339165,-3.024487,1.317365,2.689577,-0.563097,1.095719,-0.444590,2.270622
MB1_2,1.770722,-0.837442,2.005419,-0.312381,2.029168,0.591040,-0.254731,2.248437,-3.013918,1.349727,...,-5.495518,-4.143092,0.924928,-3.773692,1.082648,-2.113712,0.418748,1.161695,2.478740,1.737869


In [13]:
# Montar traços de interesse para correlação

metadata_treated = metadata_treated.copy()

# Variável binária principal para distinguir 100 nm de 1 µm
metadata_treated["is_100nm"] = (metadata_treated["particle_size_nm"] == 100).astype(int)

# Dummies por grupo para leitura mais fina no heatmap
group_dummies_treated = pd.get_dummies(metadata_treated["group"], prefix="group")

# Traits não redundantes
traits_numeric_treated = pd.concat([
    metadata_treated[[
        "particle_size_um",
        "concentration_gL",
        "is_100nm",
    ]],
    group_dummies_treated
], axis=1)

print("Traits numéricos (tratadas):", traits_numeric_treated.shape)
display(traits_numeric_treated.head())

Traits numéricos (tratadas): (21, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
sample_id,,,,,,,,,,
MA1_1,1.0,0.10,0,True,False,False,False,False,False,False
MA1_2,1.0,0.10,0,True,False,False,False,False,False,False
MA1_3,1.0,0.10,0,True,False,False,False,False,False,False
MB1_1,1.0,0.01,0,False,False,True,False,False,False,False
MB1_2,1.0,0.01,0,False,False,True,False,False,False,False


In [14]:
# Correlação módulo-traço

def safe_pearsonr(x, y):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)

    mask = x.notna() & y.notna()

    if mask.sum() < 3:
        return np.nan, np.nan

    if x[mask].nunique() < 2 or y[mask].nunique() < 2:
        return np.nan, np.nan

    r, p = pearsonr(x[mask], y[mask])
    return r, p

corr_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

pval_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

for me_col in module_eigengenes_treated.columns:
    for trait_col in traits_numeric_treated.columns:
        r, p = safe_pearsonr(
            module_eigengenes_treated[me_col],
            traits_numeric_treated[trait_col]
        )
        corr_matrix_treated.loc[me_col, trait_col] = r
        pval_matrix_treated.loc[me_col, trait_col] = p

print("Matriz de correlação:", corr_matrix_treated.shape)
print("Matriz de p-values:", pval_matrix_treated.shape)

display(corr_matrix_treated)
display(pval_matrix_treated)

Matriz de correlação: (29, 10)
Matriz de p-values: (29, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,0.571584,-0.209112,-0.571584,0.234944,-0.511549,0.225523,-0.182097,0.304732,-0.114696,0.043142
ME_bisque,-0.418720,0.068826,0.418720,-0.036964,0.153955,-0.412051,0.209405,-0.111109,0.228800,-0.032036
ME_blanchedalmond,-0.130815,0.026964,0.130815,0.184227,-0.156685,0.184992,0.296747,-0.475561,0.044939,-0.078658
ME_brown,-0.143854,0.250234,0.143854,0.125267,0.092495,0.039746,0.066520,-0.516549,0.044424,0.148097
ME_burlywood,0.279075,0.186036,-0.279075,0.205801,-0.072264,-0.292164,-0.189224,0.210712,-0.133185,0.270322
ME_chocolate,-0.074974,-0.400873,0.074974,-0.209375,-0.329048,0.103037,0.212421,-0.056243,0.222657,0.056551
ME_coral,-0.100922,0.116759,0.100922,0.081990,0.151342,0.124500,-0.068855,-0.151066,0.060239,-0.198149
ME_darkgrey,0.622239,0.216260,-0.622239,-0.057145,0.152968,0.285727,-0.315752,0.314392,-0.717194,0.337005
ME_darkorange,-0.599400,0.320878,0.599400,-0.036169,0.621779,-0.551647,0.259016,0.090450,-0.033115,-0.350314
ME_darksalmon,0.443179,-0.402970,-0.443179,-0.026380,-0.648782,0.108393,-0.012335,0.176710,0.034367,0.368027


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,0.006791,0.362977,0.006791,0.305285,0.017773,0.325640,0.429518,0.179221,0.620561,0.852696
ME_bisque,0.058866,0.766894,0.058866,0.873614,0.505219,0.063442,0.362290,0.631599,0.318470,0.890357
ME_blanchedalmond,0.571937,0.907637,0.571937,0.424049,0.497604,0.422094,0.191466,0.029338,0.846628,0.734679
ME_brown,0.533874,0.273945,0.533874,0.588485,0.690064,0.864181,0.774507,0.016508,0.848364,0.521746
ME_burlywood,0.220540,0.419434,0.220540,0.370794,0.755586,0.198744,0.411365,0.359231,0.564933,0.235962
ME_chocolate,0.746702,0.071712,0.746702,0.362359,0.145261,0.656719,0.355257,0.808668,0.331990,0.807637
ME_coral,0.663360,0.614246,0.663360,0.723853,0.512560,0.590789,0.766798,0.513338,0.795341,0.389229
ME_darkgrey,0.002595,0.346423,0.002595,0.805654,0.507987,0.209274,0.163218,0.165138,0.000253,0.135206
ME_darkorange,0.004082,0.156122,0.004082,0.876311,0.002620,0.009530,0.256895,0.696600,0.886686,0.119506
ME_darksalmon,0.044204,0.070102,0.044204,0.909629,0.001464,0.640008,0.957677,0.443522,0.882429,0.100705


In [15]:
# Correção para múltiplos testes: Benjamini-Hochberg (FDR) sobre os 140 testes
# simultâneos (14 módulos × 10 traits). Sem correção, ~7 associações falsas seriam
# esperadas ao nível α=0.05 apenas por acaso.

padj_matrix_treated = pval_matrix_treated.copy().astype(float)

for trait_col in pval_matrix_treated.columns:
    pvals = pval_matrix_treated[trait_col].values.astype(float)
    valid_mask = ~np.isnan(pvals)
    if valid_mask.sum() > 0:
        _, padj, _, _ = multipletests(pvals[valid_mask], method="fdr_bh")
        padj_col = np.full(len(pvals), np.nan)
        padj_col[valid_mask] = padj
        padj_matrix_treated[trait_col] = padj_col

print("Matriz de p-valores ajustados (BH FDR):", padj_matrix_treated.shape)
display(padj_matrix_treated)

Matriz de p-valores ajustados (BH FDR): (29, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,0.065650,0.809717,0.065650,0.9552,0.103082,0.770746,0.986382,0.519163,0.966135,0.908301
ME_bisque,0.284518,0.928008,0.284518,0.9552,0.707820,0.262833,0.986382,0.732654,0.966135,0.908301
ME_blanchedalmond,0.789818,0.958242,0.789818,0.9552,0.707820,0.840028,0.986382,0.236841,0.966135,0.908301
ME_brown,0.789818,0.794441,0.789818,0.9552,0.861395,0.895045,0.986382,0.236841,0.966135,0.865300
ME_burlywood,0.593407,0.810905,0.593407,0.9552,0.861395,0.600516,0.986382,0.704605,0.966135,0.760322
ME_chocolate,0.881802,0.693215,0.881802,0.9552,0.421257,0.840028,0.986382,0.808668,0.966135,0.908301
ME_coral,0.836411,0.848244,0.836411,0.9552,0.707820,0.840028,0.986382,0.716777,0.966135,0.865300
ME_darkgrey,0.059196,0.809717,0.059196,0.9552,0.707820,0.600516,0.986382,0.519163,0.007340,0.653494
ME_darkorange,0.059196,0.794441,0.059196,0.9552,0.037984,0.092119,0.986382,0.776976,0.966135,0.653494
ME_darksalmon,0.256385,0.693215,0.256385,0.9552,0.037984,0.840028,0.986382,0.716777,0.966135,0.653494


In [16]:
fig = px.imshow(
    corr_matrix_treated.astype(float),
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect="auto",
    labels=dict(x="Traits", y="Módulos", color="Correlação"),
    x=corr_matrix_treated.columns,
    y=corr_matrix_treated.index,
    title="Correlação entre module eigengenes e traits (somente tratadas)"
)

fig.update_layout(
    width=900, height=max(400, 35*len(corr_matrix_treated)),
    margin=dict(l=100, r=20, t=60, b=100),
    xaxis_title="Traits",
    yaxis_title="Módulos"
)

fig.show()

# Salva o heatmap como PNG (requer kaleido: pip install kaleido)
fig.write_image(str(HEATMAP_PATH))
print("Heatmap salvo em:", HEATMAP_PATH)

Heatmap salvo em: ../../data/interim/wgcna/wgcna_module_trait_heatmap_treated.png


## Consolidação dos Resultados do WGCNA

In [17]:
# Salvamento dos resultados sem controles

module_eigengenes_treated.to_csv(MODULE_EIGENGENES_TREATED_PATH)
corr_matrix_treated.to_csv(MODULE_TRAIT_CORR_PATH)
pval_matrix_treated.to_csv(MODULE_TRAIT_PVAL_PATH)
padj_matrix_treated.to_csv(MODULE_TRAIT_PADJ_PATH)

print("Arquivos salvos:")
print("-", MODULE_EIGENGENES_TREATED_PATH)
print("-", MODULE_TRAIT_CORR_PATH)
print("-", MODULE_TRAIT_PVAL_PATH)
print("-", MODULE_TRAIT_PADJ_PATH)

Arquivos salvos:
- ../../data/interim/wgcna/wgcna_module_eigengenes_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_correlations_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_pvalues_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_padj_treated.csv


In [18]:
# Resumo dos módulos mais associados aos traços de interesse.
# Inclui correlação, p-valor bruto e p-valor ajustado por BH (FDR).
# O flag `significant_*` usa padj < 0.05.

summary_rows = []

for module_name in corr_matrix_treated.index:
    row = {"module": module_name}

    for trait in ["particle_size_um", "concentration_gL", "is_100nm"]:
        if trait in corr_matrix_treated.columns:
            row[f"corr_{trait}"]        = corr_matrix_treated.loc[module_name, trait]
            row[f"pval_{trait}"]        = pval_matrix_treated.loc[module_name, trait]
            row[f"padj_{trait}"]        = padj_matrix_treated.loc[module_name, trait]
            row[f"significant_{trait}_padj005"] = padj_matrix_treated.loc[module_name, trait] < 0.05
            row[f"significant_{trait}_padj010"] = padj_matrix_treated.loc[module_name, trait] < 0.10

    summary_rows.append(row)

module_summary_df = pd.DataFrame(summary_rows)
module_summary_df.to_csv(MODULE_SUMMARY_PATH, index=False)

print("Resumo dos módulos salvo em:", MODULE_SUMMARY_PATH)
display(module_summary_df.sort_values("pval_particle_size_um", na_position="last").head(15))

Resumo dos módulos salvo em: ../../data/interim/wgcna/wgcna_module_summary.csv


,module,corr_particle_size_um,pval_particle_size_um,padj_particle_size_um,significant_particle_size_um_padj005,significant_particle_size_um_padj010,corr_concentration_gL,pval_concentration_gL,padj_concentration_gL,significant_concentration_gL_padj005,significant_concentration_gL_padj010,corr_is_100nm,pval_is_100nm,padj_is_100nm,significant_is_100nm_padj005,significant_is_100nm_padj010
7,ME_darkgrey,0.622239,0.002595,0.059196,False,True,0.216260,0.346423,0.809717,False,False,-0.622239,0.002595,0.059196,False,True
8,ME_darkorange,-0.599400,0.004082,0.059196,False,True,0.320878,0.156122,0.794441,False,False,0.599400,0.004082,0.059196,False,True
0,ME_antiquewhite,0.571584,0.006791,0.065650,False,True,-0.209112,0.362977,0.809717,False,False,-0.571584,0.006791,0.065650,False,True
19,ME_orangered,-0.480501,0.027469,0.199153,False,False,0.106015,0.647407,0.853400,False,False,0.480501,0.027469,0.199153,False,False
9,ME_darksalmon,0.443179,0.044204,0.256385,False,False,-0.402970,0.070102,0.693215,False,False,-0.443179,0.044204,0.256385,False,False
1,ME_bisque,-0.418720,0.058866,0.284518,False,False,0.068826,0.766894,0.928008,False,False,0.418720,0.058866,0.284518,False,False
11,ME_firebrick,0.333552,0.139506,0.517076,False,False,-0.283540,0.212935,0.794441,False,False,-0.333552,0.139506,0.517076,False,False
17,ME_mistyrose,-0.331082,0.142642,0.517076,False,False,0.034048,0.883513,0.958242,False,False,0.331082,0.142642,0.517076,False,False
22,ME_red,-0.302822,0.182100,0.586768,False,False,0.253125,0.268256,0.794441,False,False,0.302822,0.182100,0.586768,False,False
4,ME_burlywood,0.279075,0.220540,0.593407,False,False,0.186036,0.419434,0.810905,False,False,-0.279075,0.220540,0.593407,False,False


## Nota Metodológica — Threshold de Significância (α)

**Para discussão com o grupo:**

Os testes de associação módulo-trait foram corrigidos para múltiplos testes pelo método Benjamini-Hochberg (FDR). Com n=21 amostras tratadas e o número de módulos gerado, nenhum módulo atingiu o threshold convencional de **padj < 0.05**.

Contudo, em estudos exploratórios de genômica com amostras pequenas, **α = 0.10 (FDR 10%) é aceito na literatura** como critério para seleção de módulos candidatos. Isso significa: entre todos os resultados chamados significativos, esperamos no máximo 10% de falsos positivos.

**Decisão a tomar antes de prosseguir para enriquecimento funcional:**
- Usar **α = 0.05** → nenhum módulo FDR-significativo; análise downstream baseada em significância nominal (p < 0.05 bruto)
- Usar **α = 0.10** → módulos com `padj < 0.10` são considerados significativos; justificativa metodológica deve ser explicitada na seção de métodos

Os flags `significant_*_padj005` e `significant_*_padj010` estão ambos disponíveis em `wgcna_module_summary.csv` para facilitar essa decisão.

## Comparação de Parâmetros

In [19]:
# Comparação entre a configuração original (minModuleSize=50, MEDissThres=0.2)
# arquivada em v_minmod50/ e a configuração atual definida nas constantes deste notebook.
# Execute esta célula após rodar o notebook completo com os novos parâmetros.

import pandas as pd

BASELINE_DIR = WGCNA_DIR / "v_minmod50"
CURRENT_LABEL = f"minmod{MIN_MODULE_SIZE} / meds{int(ME_DISS_THRES * 100)} (atual)"
BASELINE_LABEL = "minmod50 / meds20 (original)"


def summarize_run(label, run_dir):
    summary_path = run_dir / "wgcna_module_summary.csv"
    modules_path = run_dir / "wgcna_gene_modules.csv"

    if not summary_path.exists() or not modules_path.exists():
        return {"versão": label, "nº módulos": "—", "melhor p-val particle_size": "—",
                "melhor padj particle_size": "—", "módulos padj < 0.05": "—"}

    summary = pd.read_csv(summary_path)
    modules = pd.read_csv(modules_path)

    n_modules = modules["module"].nunique()

    best_pval = summary["pval_particle_size_um"].min()
    best_padj = summary["padj_particle_size_um"].min()
    n_sig_005 = (summary["padj_particle_size_um"] < 0.05).sum()
    n_sig_010 = (summary["padj_particle_size_um"] < 0.10).sum()

    return {
        "versão": label,
        "nº módulos": n_modules,
        "melhor p-val particle_size": f"{best_pval:.4f}",
        "melhor padj particle_size": f"{best_padj:.4f}",
        "módulos padj < 0.05": n_sig_005,
        "módulos padj < 0.10": n_sig_010,
    }


rows = [
    summarize_run(BASELINE_LABEL, BASELINE_DIR),
    summarize_run(CURRENT_LABEL, WGCNA_DIR),
]

comparison_df = pd.DataFrame(rows).set_index("versão")
print("=== Comparação de parâmetros WGCNA ===")
display(comparison_df.T)

def list_sig_modules(label, run_dir, alpha=0.10):
    summary_path = run_dir / "wgcna_module_summary.csv"
    if not summary_path.exists():
        print(f"\n[{label}] wgcna_module_summary.csv não encontrado.")
        return
    summary = pd.read_csv(summary_path)
    if "padj_particle_size_um" not in summary.columns:
        print(f"\n[{label}] coluna padj_particle_size_um ausente.")
        return
    sig = (
        summary[summary["padj_particle_size_um"] < alpha]
        [["module", "corr_particle_size_um", "pval_particle_size_um", "padj_particle_size_um"]]
        .rename(columns={
            "module": "módulo",
            "corr_particle_size_um": "r",
            "pval_particle_size_um": "p-val",
            "padj_particle_size_um": "padj",
        })
        .sort_values("p-val")
        .reset_index(drop=True)
    )
    sig["r"]      = sig["r"].round(3)
    sig["p-val"]  = sig["p-val"].round(4)
    sig["padj"]   = sig["padj"].round(4)
    sig["direção"] = sig["r"].apply(lambda r: "↑ em 1µm" if r > 0 else "↑ em 100nm")
    print(f"\n=== Módulos com padj < {alpha} — {label} ===")
    display(sig)

list_sig_modules(BASELINE_LABEL, BASELINE_DIR)
list_sig_modules(CURRENT_LABEL, WGCNA_DIR)

=== Comparação de parâmetros WGCNA ===


versão,minmod50 / meds20 (original),minmod20 / meds25 (atual)
nº módulos,14,29
melhor p-val particle_size,0.0138,0.0026
melhor padj particle_size,0.0929,0.0592
módulos padj < 0.05,0,0
módulos padj < 0.10,3,3



=== Módulos com padj < 0.1 — minmod50 / meds20 (original) ===


,módulo,r,p-val,padj,direção
0,ME_darkred,-0.528,0.0138,0.0929,↑ em 100nm
1,ME_lightgrey,-0.523,0.0149,0.0929,↑ em 100nm
2,ME_darkgrey,0.504,0.0199,0.0929,↑ em 1µm



=== Módulos com padj < 0.1 — minmod20 / meds25 (atual) ===


,módulo,r,p-val,padj,direção
0,ME_darkgrey,0.622,0.0026,0.0592,↑ em 1µm
1,ME_darkorange,-0.599,0.0041,0.0592,↑ em 100nm
2,ME_antiquewhite,0.572,0.0068,0.0657,↑ em 1µm
